# XMM-Newton 单星系 CGM 处理流程（组会讲稿用）

**数据**

XMM-Newton EPIC：MOS1、MOS2、PN 三个探测器。

图像已在天空平面上对齐，像素尺度一致（本脚本用固定角秒/像素）。

**要做什么**

先把不可靠像素和亮污染mask掉。

再在环带里把三个探测器的信号按物理量加起来，做径向轮廓和简单方向分解。

最终关心的是：**星系外围弱辐射能不能稳定测到**，而不是单张好看的彩图。

**读图习惯**

脚本生成的图里，坐标轴和图例是英文。

这份笔记用中文写，方便口头讲。

---

## 1. 输入：文件名在说什么

**模板**

从合并图文件名入手，例如：`comb-fovimsky-....fits`。

把开头的 `comb-` 换成探测器名，就得到这一台探测器的同一场 `fovimsky`。

**同一套网格上的配套文件**

只改文件名里的关键词，其余不变：

- `fovimsky`：观测强度（计数率）

- `bkgimsky`：背景模型

- `protimsky`：点源/紧凑源模型，要从观测里减掉

- `expimsky`：曝光，用来做归一化和低曝光切除

- `maskimsky`：CCD 有效区。约定是 **1 保留，0 丢掉**

**Cheese**

每台探测器一个：`{探测器}-cheeset.fits`。

含义：哪些像素在科学上可信（具体定义看你管线）。脚本里把 **四舍五入等于 1** 当保留。

**DS9 区域**

主 `.reg`：画星系椭圆、画背景参考环（annulus）。

可选 `_out.reg`：手动画掉邻近亮源、坏边之类。

若 DS9 用 1 起始像素，脚本里会把坐标整体平移一格，对齐 FITS 的 0 起始。

---

## 2. 总流程：先想清楚顺序

下面是一条线从左到右，**顺序不能乱**。

前面的洞和坏点，会直接影响后面的背景统计和自动mask。

```mermaid
flowchart LR
  A[Cheese + CCD] --> B[减背景与点源]
  B --> C[曝光阈值]
  C --> D[高斯平滑]
  D --> E[估背景与涨落]
  E --> F[连通域 + 双阈值]
  F --> G[膨胀与最终规则]
  G --> H[每探测器 mask]
  H --> I[环内求和]
  I --> J[径向 profile]
```

**口头版十步**

1. 用 Cheese 和 CCD mask 先把“根本不该看的像素”清掉。

2. 观测减背景，再减点源模型，得到净计数。

3. 曝光太低的像素一律不要，避免噪声被放大。

4. 对表面亮度做平滑，噪声碎点会少很多。

5. 在 annulus 里估一个“正常背景有多高、自然涨落有多大”。

6. 把每个像素换成“高出背景多少个涨落单位”。

7. 用双阈值 + 连通域，把连成片的高值当成弥散污染扣掉。

8. 稍微膨胀一下mask，吃掉源边上的小晕。

9. 加上星系椭圆、极高值等规则，得到 **final mask**。

10. 三个探测器各自一张 mask，再在环里把 net、方差、曝光 **逐像素相加**，做 profile。

**合并图**

脚本 **不画** 三探测器合并后的表面亮度总图。

合并只发生在：**径向环里**，对 net、方差、曝光做求和，再算轮廓。

---

## 3. 掩膜：每一步在挡什么

**Cheese 和 CCD**

最先做。

不通过的像素，在 fov、bkg、点源模型、曝光上 **一起** 变成无效值。

这样后面的减法和统计，不会吃到芯片缝、无效列那种东西。

**手绘 `_out.reg`**

自动mask总有盲区。

旁边亮星系、明显条纹之类，用区域文件一次性盖住。

---

## 4. 从计数到“能看的表面亮度”

**净信号**

观测里减掉背景，再减掉点源模型，剩下的是净计数率。

**噪声量级（脚本里的做法）**

把观测、背景、点源模型三者的计数率 **相加再开方**，当作这一像素噪声的粗略尺度。

这是常见近似。若你上游已经给了更严格的误差表，可以以后替换。

**表面亮度**

净计数除以曝光，再除以像素对应的立体角，得到每平方度、每秒的量纲。

后面画 profile 时，会再换算到 **每平方角分** 等展示单位。

**曝光切割**

曝光低于固定阈值的像素全部扔掉。

理由很简单：那里怎么减背景都抖，profile 会被尾巴拖脏。

用固定阈值，比按分位数偷偷裁一刀 **更好解释**。

---

## 5. 平滑：让“形状”露出来

平滑用高斯核。

图上有洞（NaN）时，脚本用 **带权重的卷积**：只让有效像素参与归一化。

这样洞边缘不会被零填平拉出一条假边。

---

## 6. 背景标定：在哪里估“正常有多高”

只在 **annulus 环** 里取像素。

可选：把星系椭圆里的像素从背景统计里拿掉，避免盘光抬高背景。

再可选：先去掉特别离谱的极高值，以免未扣干净的亮核把尺度拉爆。

平滑图上取 **中位数** 当背景中心。

用 **MAD** 估自然涨落，比标准差耐异常点。

每个像素写成：**比背景高多少个涨落单位**。后面所有阈值都在这个无量纲场上做。

---

## 7. 弥散污染：双阈值 + 连通域

**为什么不用单阈值**

单阈值要么碎成满天星，要么拦不住连成片的晕。

**双阈值在干什么**

先允许一片“略高”的像素连成区域。

只有这片里 **至少有一个很高** 的种子点，整片才保留为污染。

孤立噪声点通常过不了这一关。

**内圈和外圈两套参数**

以固定半径为界：内圈结构乱，阈值严一点；外圈相对平，阈值松一点。

**面积下限**

太小的斑块直接扔，多半是热像素。

**膨胀**

对mask做几次形态学膨胀，把源边上一圈弱晕也盖住。

**可选：负值区域**

若打开，对称地处理大尺度“减过了头”的斑块。

默认关。若以后打开，注意脚本里阈值变量名是否写对，避免运行时报错。

**final mask 还加什么**

不在 hot mask 里、平滑场有效、可选不在星系椭圆里、可选去掉硬上限以上的极端像素。

通过则记为 1，否则 0。并写出每探测器的 `*_mask.fits`。

---

## 8. 三探测器合并：只在环里做加法

每个探测器用自己的 final mask。

对 **净计数、方差、曝光** 三个量，在像素上分别求和。

这不是把三张表面亮度图平均成一张展示图。

而是：**同一物理像素上，把三台仪器的计数和误差预算加在一起**。

合并后的“表面亮度”只作诊断（例如环内均值、有多少负像素），**不当主结果图**。

主结果是：**用 net、方差、曝光的和** 算出来的径向轮廓。

背景参考线在 annulus 里对合并诊断图取平均，再换单位，画成水平虚线。

---

## 9. 径向 profile：箱宽怎么定

先在给定半径范围内挑出有效像素。

按半径从小到大排序。

从内向外累加净计数和方差。

累加到 **累计信噪比** 达到设定门槛，就切一刀，开始下一个径向箱。

这样做的直接好处：

外圈每像素很弱，自动用 **更宽的径向箱** 换稳定度。

内圈像素多、信号相对强，箱可以细一些。

每个箱里仍用 **净计数之和除以曝光之和** 再换单位，得到该半径的平均表面亮度。

误差条用方差之和开方，同样归一。

横轴除角分外，还用距离模算一层 kpc，方便和光学论文对比。

---

## 10. 方向分解和阶段直方图

**方向**

用星系椭圆的长轴方向做参考，把平面划成“沿盘”和“垂直盘”两套扇区。

在每个径向箱里，对两套扇区分别做 **同样的 net / 方差 / 曝光求和**。

散点示意底图用的是 **第一台探测器** 的表面亮度，只是直观展示扇区落在哪，不是合并科学图。

汇报时建议对照图例，核对脚本里扇区变量是否和口头命名一致。

**阶段直方图**

从原始到最终，分多步画像素值分布。

每一步都对比：**全图** 和 **外圈子样本**。

用来肉眼看：mask 是否过头、外圈是否被内区长尾污染、某一步是否引入奇怪形状。

---

## 11. 主要产出（对着文件夹讲）

- 每探测器：平滑图、污染轮廓图、final mask 图、六阶段直方图

- 每探测器：`mask.fits`（uint8，1 为保留）

- 合并：径向 profile（角分、kpc 各一张）

- 合并：方向 profile（角分、kpc）

- 扇区示意图一张

---

## 12. 组会可强调的四句

1. 掩膜顺序固定、可复述：先几何与探测器有效区，再物理减除，再统计mask。

2. 背景尺度在环里用中位数和 MAD 估，抗尚未扣干净的弱结构。

3. 弥散污染用 **连通域 + 双阈值 + 膨胀**，针对成片晕而不是单像素。

4. 合并结果以 **环内求和后的 profile** 为主，避免误导性的“拼一张合并 SB 图”。

---

**跑代码**

主程序放在 `.py` 里，在下方单元格用 `%run` 指向你的路径即可。

本笔记本只负责讲清楚 **做了什么、为什么**，与代码分开放，改参数不影响讲稿结构。

In [ ]:
# %matplotlib inline
# %run /path/to/your_xmm_cgm_pipeline.py